In [ ]:
!pip install -q transformers datasets evaluate scikit-learn accelerate peft sentencepiece protobuf
!pip install -U "torchao>=0.16.0" peft transformers datasets evaluate

In [ ]:
import os
import random
import numpy as np
import pandas as pd

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_PROJECT"] = "Fact_Checker_Travel"

import torch
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, 
                          TrainingArguments, Trainer, DataCollatorWithPadding, 
                          pipeline, EarlyStoppingCallback)
from peft import get_peft_model, LoraConfig, TaskType,PromptTuningConfig, PromptTuningInit
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score


SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")

def reset_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

try:
    import wandb
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    
    REPORT_TO = "wandb"
    print("W&B attivato.")
except Exception as e:
    print(f"W&B disabilitato. ({e})")
    os.environ["WANDB_DISABLED"] = "true"
    REPORT_TO = "none"

In [ ]:
from datasets import Dataset, DatasetDict, ClassLabel

df = pd.read_csv('/kaggle/input/datasets/gabrieelee/factcheck-dataset-japan-csv/factcheck_dataset_japan.csv')
df = df.rename(columns={"context": "premise", "claim": "hypothesis"})

ID2LABEL = {0: "contradiction", 1: "entailment", 2: "neutral_hard"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = len(ID2LABEL)

dataset = Dataset.from_pandas(df)
train_test = dataset.train_test_split(test_size=0.3, seed=SEED)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=SEED)
ds = DatasetDict({'train': train_test['train'], 'validation': val_test['train'], 'test': val_test['test']})

rows = []
for split in ["train", "validation", "test"]:
    counts = pd.Series(ds[split]["label"]).value_counts().sort_index()
    row = {ID2LABEL[i]: int(counts.get(i, 0)) for i in ID2LABEL}   
    row["Totale"] = len(ds[split])
    rows.append(row)

dist = pd.DataFrame(rows, index=["train", "validation", "test"])
dist.loc["TOTALE"] = dist.sum()

print("\nConteggio esempi per split e per label:")
print(dist)

train_labels = ds['train']['label']
counts = pd.Series(train_labels).value_counts().sort_index().values.astype(float)
class_weights = torch.tensor(counts.sum() / (NUM_LABELS * counts), dtype=torch.float).to(DEVICE)
print("\n\n Pesi calcolati per le classi (Contradiction, Entailment, Neutral):", class_weights.tolist())

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits.float(), labels, weight=class_weights.float())
        return (loss, outputs) if return_outputs else loss

In [ ]:
model_checkpoint = "/kaggle/input/datasets/gabrieelee/mdeberta-v3-base-mnli-xnli-offline"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(
        examples["premise"], 
        examples["hypothesis"], 
        truncation=True, 
        max_length=256
    )

tokenized_ds = ds.map(tokenize_function, batched=True)
colonne_da_rimuovere = [c for c in ["fact_id", "premise", "hypothesis", "claim_type", "__index_level_0__"] if c in tokenized_ds["train"].column_names]
tokenized_ds = tokenized_ds.remove_columns(colonne_da_rimuovere)
tokenized_ds.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

import evaluate
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.logits if hasattr(eval_pred, 'logits') else eval_pred.predictions, axis=-1)
    acc = metric_acc.compute(predictions=preds, references=eval_pred.label_ids)["accuracy"]
    f1 = metric_f1.compute(predictions=preds, references=eval_pred.label_ids, average="macro")["f1"]
    return {"accuracy": acc, "macro_f1": f1}

In [ ]:
print("--- CALCOLO BASELINE SUL TEST SET ---")

nli_pipe = pipeline("text-classification", model="/kaggle/input/datasets/gabrieelee/mdeberta-v3-base-mnli-xnli-offline", device=0 if DEVICE=="cuda" else -1)

def map_nli_label(label_str):
    if "entailment" in label_str.lower(): return 1
    if "contradiction" in label_str.lower(): return 0
    return 2

zero_shot_preds, true_labels = [], []
for item in ds["test"]:
    text = f"{item['premise']} {nli_pipe.tokenizer.sep_token} {item['hypothesis']}"
    res = nli_pipe(text, truncation=True, max_length=256)[0]
    zero_shot_preds.append(map_nli_label(res['label']))
    true_labels.append(item['label'])

zs_acc = metric_acc.compute(predictions=zero_shot_preds, references=true_labels)["accuracy"]
zs_f1 = metric_f1.compute(predictions=zero_shot_preds, references=true_labels, average="macro")["f1"]
print(f"Baseline (Zero-Shot mDeBERTa) - Accuracy: {zs_acc:.4f}, Macro-F1: {zs_f1:.4f}")

del nli_pipe
torch.cuda.empty_cache()

In [ ]:

print("\n Avvio Ricerca Iperparametri LoRA sul Validation Set (Con Early Stopping)...")

results_log = []
lora_ranks = [8,16]
learning_rates = [1e-4,2e-4, 3e-4]

best_f1 = 0
best_config = {}

for r in lora_ranks:
    for lr in learning_rates:
        print(f"\n Test -> Rank: {r}, LR: {lr}")
        reset_seeds(SEED)
        lora_config = LoraConfig(
            r=r, 
            lora_alpha=16, 
            lora_dropout=0.2, 
            bias="none", 
            task_type=TaskType.SEQ_CLS,
            target_modules=["query_proj", "key_proj", "value_proj"]
        )
        
        temp_model = AutoModelForSequenceClassification.from_pretrained( model_checkpoint, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID)
        temp_model = get_peft_model(temp_model, lora_config)
        training_args = TrainingArguments(
            output_dir=f"./lora_search_r{r}_lr{lr}", 
            learning_rate=lr,
            per_device_train_batch_size=8, 
            per_device_eval_batch_size=8,
            num_train_epochs=10,                      
            eval_strategy="epoch", 
            save_strategy="epoch",                   
            load_best_model_at_end=True,             
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            report_to="none" 
        )
        
        temp_trainer = WeightedTrainer(
            model=temp_model, args=training_args,
            train_dataset=tokenized_ds["train"], eval_dataset=tokenized_ds["validation"],
            processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] 
        )
        
        temp_trainer.train()
        
        val_result = temp_trainer.evaluate()
        val_f1 = val_result['eval_macro_f1']
        
        results_log.append({
            "Rank": r, "LR": lr, 
            "Epoca_Fermata": temp_trainer.state.epoch, 
            "Val_Accuracy": val_result['eval_accuracy'], 
            "Val_Macro_F1": val_f1
        })
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_config = {'r': r, 'lr': lr}

print(f"\n Miglior configurazione: {best_config} (Val Macro F1: {best_f1:.4f})")
display(pd.DataFrame(results_log).sort_values("Val_Macro_F1", ascending=False))

In [ ]:
print(f"\n Avvio addestramento finale con Rank {best_config['r']} e LR {best_config['lr']}...")

final_lora_config = LoraConfig(
    r=best_config['r'], 
    lora_alpha=16, 
    lora_dropout=0.2, 
    bias="none", 
    task_type=TaskType.SEQ_CLS,
    target_modules=["query_proj", "key_proj", "value_proj"]
)

final_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID)
final_model = get_peft_model(final_model, final_lora_config)
final_training_args = TrainingArguments(
    output_dir="./lora_best_model",
    learning_rate=best_config['lr'],
    per_device_train_batch_size=8, 
    per_device_eval_batch_size=8,
    num_train_epochs=15, 
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    weight_decay=0.01,         
    warmup_ratio=0.1,
    report_to=REPORT_TO
)

final_trainer = WeightedTrainer(
    model=final_model, args=final_training_args,
    train_dataset=tokenized_ds["train"], eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] 
)
lora_trainable_params = final_model.num_parameters(only_trainable=True)
lora_total_params = final_model.num_parameters()
print(f"LoRA - trainable: {lora_trainable_params:,} | total: {lora_total_params:,} | trainable%: {lora_trainable_params/lora_total_params*100:.2f}%")

final_trainer.train()
print(f"Il miglior checkpoint è stato caricato automaticamente in memoria grazie a load_best_model_at_end.")

print("\n Valutazione sul TEST SET con il miglior modello...")
final_test_result = final_trainer.evaluate(eval_dataset=tokenized_ds["test"])

print(f"Risultati sul Test Set -> Accuracy: {final_test_result['eval_accuracy']:.4f} | Macro F1: {final_test_result['eval_macro_f1']:.4f}")

In [ ]:
print("\n Metriche dettagliate per Zero-Shot (baseline)")


prec_macro_zs = precision_score(true_labels, zero_shot_preds, average='macro')
rec_macro_zs = recall_score(true_labels, zero_shot_preds, average='macro')
f1_macro_zs = f1_score(true_labels, zero_shot_preds, average='macro')

prec_weighted_zs = precision_score(true_labels, zero_shot_preds, average='weighted')
rec_weighted_zs = recall_score(true_labels, zero_shot_preds, average='weighted')
f1_weighted_zs = f1_score(true_labels, zero_shot_preds, average='weighted')

print(f"Accuracy: {zs_acc:.4f}")
print(f"Precision (macro): {prec_macro_zs:.4f} | Recall (macro): {rec_macro_zs:.4f} | F1 (macro): {f1_macro_zs:.4f}")
print(f"Precision (weighted): {prec_weighted_zs:.4f} | Recall (weighted): {rec_weighted_zs:.4f} | F1 (weighted): {f1_weighted_zs:.4f}")


print("\n Classification Report per classe (Zero-Shot):")
print(classification_report(true_labels, zero_shot_preds, target_names=ID2LABEL.values()))


cm_zs = confusion_matrix(true_labels, zero_shot_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm_zs, annot=True, fmt='d', cmap='Oranges',xticklabels=ID2LABEL.values(), yticklabels=ID2LABEL.values())
plt.title("Matrice di confusione - Zero-Shot")
plt.xlabel("Predetto")
plt.ylabel("Vero")
plt.show()

In [ ]:

print("\n Metriche dettagliate per LoRA sul test set")

predictions_lora = final_trainer.predict(tokenized_ds["test"])
preds_lora = np.argmax(predictions_lora.predictions, axis=-1)
refs = np.array(ds["test"]["label"])

acc_lora = accuracy_score(refs, preds_lora)
prec_macro_lora = precision_score(refs, preds_lora, average='macro')
rec_macro_lora = recall_score(refs, preds_lora, average='macro')
f1_macro_lora = f1_score(refs, preds_lora, average='macro')

prec_weighted_lora = precision_score(refs, preds_lora, average='weighted')
rec_weighted_lora = recall_score(refs, preds_lora, average='weighted')
f1_weighted_lora = f1_score(refs, preds_lora, average='weighted')

print(f"LoRA - trainable: {lora_trainable_params:,} | total: {lora_total_params:,} | trainable%: {lora_trainable_params/lora_total_params*100:.2f}%")
print(f"Accuracy: {acc_lora:.4f}")
print(f"Precision (macro): {prec_macro_lora:.4f} | Recall (macro): {rec_macro_lora:.4f} | F1 (macro): {f1_macro_lora:.4f}")
print(f"Precision (weighted): {prec_weighted_lora:.4f} | Recall (weighted): {rec_weighted_lora:.4f} | F1 (weighted): {f1_weighted_lora:.4f}")

print("\n Classification Report per classe (LoRA):")
print(classification_report(refs, preds_lora, target_names=ID2LABEL.values()))

cm_lora = confusion_matrix(refs, preds_lora)
plt.figure(figsize=(6,5))
sns.heatmap(cm_lora, annot=True, fmt='d', cmap='Blues',
            xticklabels=ID2LABEL.values(), yticklabels=ID2LABEL.values())
plt.title("Matrice di confusione - LoRA")
plt.xlabel("Predetto")
plt.ylabel("Vero")
plt.show()

In [ ]:
print("\n Valutazione sul TEST SET...")
final_test_result = final_trainer.evaluate(eval_dataset=tokenized_ds["test"])

print("\n--- CONFRONTO FINALE ---")
result_comparison = {
    "Baseline: Zero-Shot NLI": {"Accuracy": zs_acc, "Macro F1": zs_f1},
    f"LoRA (r={best_config['r']}, lr={best_config['lr']})": {"Accuracy": final_test_result['eval_accuracy'], "Macro F1": final_test_result['eval_macro_f1']}
}
display(pd.DataFrame(result_comparison).T)


print("\n--- ERROR ANALYSIS (Analisi di 3 Fallimenti) ---")


predictions_output = final_trainer.predict(tokenized_ds["test"])
preds = np.argmax(predictions_output.predictions, axis=-1)


refs = np.array(ds["test"]["label"])


errors_idx = np.where(preds != refs)[0]

for i in errors_idx[:3]:
    original = ds["test"][int(i)]
    print("-" * 60)
    print(f"CONTESTO (Premise): {original['premise']}")
    print(f"CLAIM (Hypothesis): {original['hypothesis']}")
    print(f"ETICHETTA VERA: {ID2LABEL[refs[i]]} | PREDETTO DAL MODELLO: {ID2LABEL[preds[i]]}")


if REPORT_TO == "wandb":
    wandb.finish()

In [ ]:
def create_ptuning_model(num_virtual_tokens=20, learning_rate=1e-3):
    """
    Crea un modello per sequence classification con p‑tuning.
    """
    pt_config = PromptTuningConfig(
        task_type=TaskType.SEQ_CLS,
        prompt_tuning_init=PromptTuningInit.RANDOM,   
        num_virtual_tokens=num_virtual_tokens,
        prompt_tuning_init_text="Determina la relazione logica tra il seguente contesto e l'affermazione:",    
        tokenizer_name_or_path=model_checkpoint,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )
    model = get_peft_model(model, pt_config)
    return model

In [ ]:
print("\n🔍 Avvio ricerca iperparametri per P‑Tuning (sul validation set)...")

pt_results = []
pt_ranks = [8,16,25, 30, 50]         
pt_lrs = [1e-3, 3e-3, 1e-2]    

best_pt_f1 = 0
best_pt_config = {}

for num_tokens in pt_ranks:
    for lr in pt_lrs:
        print(f"\n Test P‑Tuning -> Tokens: {num_tokens}, LR: {lr}")
        reset_seeds(SEED)

        
        model_pt = create_ptuning_model(num_virtual_tokens=num_tokens)
        
        
        training_args_pt = TrainingArguments(
            output_dir=f"./pt_search_t{num_tokens}_lr{lr}",
            learning_rate=lr,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            num_train_epochs=30,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            report_to="none"
        )

        trainer_pt = WeightedTrainer(
            model=model_pt,
            args=training_args_pt,
            train_dataset=tokenized_ds["train"],
            eval_dataset=tokenized_ds["validation"],
            processing_class=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
        )

        trainer_pt.train()

        
        val_result_pt = trainer_pt.evaluate()
        val_f1_pt = val_result_pt["eval_macro_f1"]

        pt_results.append({
            "Virtual Tokens": num_tokens,
            "LR": lr,
            "Epoca_Fermata": trainer_pt.state.epoch,
            "Val_Accuracy": val_result_pt["eval_accuracy"],
            "Val_Macro_F1": val_f1_pt
        })

        if val_f1_pt > best_pt_f1:
            best_pt_f1 = val_f1_pt
            best_pt_config = {"num_tokens": num_tokens, "lr": lr}

print(f"\n Miglior configurazione P‑Tuning: {best_pt_config} (Val Macro F1: {best_pt_f1:.4f})")
display(pd.DataFrame(pt_results).sort_values("Val_Macro_F1", ascending=False))

In [ ]:
print(f"\n Addestramento finale P‑Tuning con {best_pt_config['num_tokens']} token e LR {best_pt_config['lr']}...")


final_pt_model = create_ptuning_model(num_virtual_tokens=best_pt_config["num_tokens"])

final_training_args_pt = TrainingArguments(
    output_dir="./pt_best_model",
    learning_rate=best_pt_config["lr"],
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=30,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    weight_decay=0.01,
    warmup_ratio=0.1,
    report_to=REPORT_TO          
)

final_trainer_pt = WeightedTrainer(
    model=final_pt_model,
    args=final_training_args_pt,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

pt_trainable_params = final_pt_model.num_parameters(only_trainable=True)
pt_total_params = final_pt_model.num_parameters()
print(f"P‑Tuning - trainable: {pt_trainable_params:,} | total: {pt_total_params:,} | trainable%: {pt_trainable_params/pt_total_params*100:.2f}%")
final_trainer_pt.train()

print("\n Valutazione sul TEST SET con p‑tuning...")
final_test_result_pt = final_trainer_pt.evaluate(eval_dataset=tokenized_ds["test"])

print(f"Risultati P‑Tuning sul Test Set -> Accuracy: {final_test_result_pt['eval_accuracy']:.4f} | Macro F1: {final_test_result_pt['eval_macro_f1']:.4f}")

In [ ]:
# --- Valutazione per P-Tuning ---
print("\n Metriche dettagliate per P‑Tuning sul test set")

predictions_pt = final_trainer_pt.predict(tokenized_ds["test"])
preds_pt = np.argmax(predictions_pt.predictions, axis=-1)

acc_pt = accuracy_score(refs, preds_pt)
prec_macro_pt = precision_score(refs, preds_pt, average='macro')
rec_macro_pt = recall_score(refs, preds_pt, average='macro')
f1_macro_pt = f1_score(refs, preds_pt, average='macro')

prec_weighted_pt = precision_score(refs, preds_pt, average='weighted')
rec_weighted_pt = recall_score(refs, preds_pt, average='weighted')
f1_weighted_pt = f1_score(refs, preds_pt, average='weighted')

print(f"P‑Tuning - trainable: {pt_trainable_params:,} | total: {pt_total_params:,} | trainable%: {pt_trainable_params/pt_total_params*100:.2f}%")
print(f"Accuracy: {acc_pt:.4f}")
print(f"Precision (macro): {prec_macro_pt:.4f} | Recall (macro): {rec_macro_pt:.4f} | F1 (macro): {f1_macro_pt:.4f}")
print(f"Precision (weighted): {prec_weighted_pt:.4f} | Recall (weighted): {rec_weighted_pt:.4f} | F1 (weighted): {f1_weighted_pt:.4f}")

print("\n Classification Report per classe (P‑Tuning):")
print(classification_report(refs, preds_pt, target_names=ID2LABEL.values()))

cm_pt = confusion_matrix(refs, preds_pt)
plt.figure(figsize=(6,5))
sns.heatmap(cm_pt, annot=True, fmt='d', cmap='Greens',
            xticklabels=ID2LABEL.values(), yticklabels=ID2LABEL.values())
plt.title("Matrice di confusione - P‑Tuning")
plt.xlabel("Predetto")
plt.ylabel("Vero")
plt.show()

In [ ]:
print("\n--- CONFRONTO FINALE TRA APPROCCI ---")

comparison = {
    "Zero-Shot": {
        "Accuracy": zs_acc,
        "Precision (macro)": prec_macro_zs,
        "Recall (macro)": rec_macro_zs,
        "F1 (macro)": f1_macro_zs,
        "Precision (weighted)": prec_weighted_zs,
        "Recall (weighted)": rec_weighted_zs,
        "F1 (weighted)": f1_weighted_zs,
    },
    "LoRA": {
        "Accuracy": acc_lora,
        "Precision (macro)": prec_macro_lora,
        "Recall (macro)": rec_macro_lora,
        "F1 (macro)": f1_macro_lora,
        "Precision (weighted)": prec_weighted_lora,
        "Recall (weighted)": rec_weighted_lora,
        "F1 (weighted)": f1_weighted_lora,
    },
    "P‑Tuning": {
        "Accuracy": acc_pt,
        "Precision (macro)": prec_macro_pt,
        "Recall (macro)": rec_macro_pt,
        "F1 (macro)": f1_macro_pt,
        "Precision (weighted)": prec_weighted_pt,
        "Recall (weighted)": rec_weighted_pt,
        "F1 (weighted)": f1_weighted_pt,
    }
}

# Converti in DataFrame e mostra
df_comparison = pd.DataFrame(comparison).T.round(4)
display(df_comparison)

In [ ]:
print("\n--- CONFRONTO FINALE TRA APPROCCI ---")
comparison = {
    "Baseline Zero-Shot": {"Accuracy": zs_acc, "Macro F1": zs_f1},
    f"LoRA (r={best_config['r']}, lr={best_config['lr']})": {
        "Accuracy": final_test_result['eval_accuracy'],
        "Macro F1": final_test_result['eval_macro_f1']
    },
    f"P‑Tuning (tokens={best_pt_config['num_tokens']}, lr={best_pt_config['lr']})": {
        "Accuracy": final_test_result_pt['eval_accuracy'],
        "Macro F1": final_test_result_pt['eval_macro_f1']
    }
}
display(pd.DataFrame(comparison).T)

In [ ]:

print("\n--- SUCCESS ANALYSIS (Esempi classificati correttamente) ---")

logits = predictions_output.predictions
probs = np.exp(logits - logits.max(axis=-1, keepdims=True))
probs = probs / probs.sum(axis=-1, keepdims=True)

correct_idx = np.where(preds == refs)[0]

shown = []
for cls in ID2LABEL:                          
    cand = [i for i in correct_idx if refs[i] == cls]
    if cand:
        best = max(cand, key=lambda i: probs[i][cls])
        shown.append(best)

for i in shown:
    original = ds["test"][int(i)]
    conf = probs[i][preds[i]] * 100
    print("-" * 60)
    print(f"CONTESTO (Premise): {original['premise']}")
    print(f"CLAIM (Hypothesis): {original['hypothesis']}")
    print(f"ETICHETTA VERA: {ID2LABEL[refs[i]]} | PREDETTO: {ID2LABEL[preds[i]]} "
          f"(confidenza: {conf:.1f}%)")

print("-" * 60)
print(f"Totale corretti sul test: {len(correct_idx)}/{len(refs)} "
      f"({len(correct_idx)/len(refs)*100:.1f}%)")